# TP 03 — Évaluer et optimiser un système RAG

**Les carnets du RAG · Master 2 — Industry 4.0 / Systèmes informatiques pour le génie de la logistique industrielle et des services**

**Durée : 2 heures · Exécution recommandée : Google Colab (CPU suffisant) · Aucune clé API nécessaire.**

## Objectifs
1. Mesurer **Precision@k**, **Recall@k**, **Hit@k**, **RR/MRR** sur un ensemble annoté.
2. Comparer quatre variantes : **A** dense, **B** autre chunking, **C** hybride BM25+dense, **D** hybride avec réordonnancement interprétable.
3. Diagnostiquer les erreurs par question, sans sélectionner une variante au seul vu d'une moyenne.
4. Auditer séparément les affirmations d'une réponse générée, puis rédiger une conclusion prudente.

> Les documents sont fictifs. Le TP mesure des retrievers réels mais ne prétend pas valider une solution industrielle. Les réponses rédigées dans l'audit sont des exemples à annoter, et non des générations LLM.

## Étape 0 — Préparer l’environnement (5 min)
Nous installons un encodeur multilingue, FAISS pour la recherche exacte sur vecteurs normalisés et BM25 pour la recherche lexicale. RRF sera implémentée manuellement pour rendre le calcul explicite. Les versions Python évoluent : si une installation échoue, actualiser les versions ou consulter les notebooks précédents.

In [ ]:
%pip -q install 'sentence-transformers>=3,<6' 'faiss-cpu>=1.8' 'rank-bm25>=0.2.2' pandas numpy matplotlib

## Étape 1 — Charger le même corpus et constituer un jeu de référence (10 min)
Le corpus pédagogique contient **12 documents** et **14 questions positives** : les 12 questions du TP 1 plus deux questions multisources. Nous ajoutons un cas **sans preuve documentaire** ; il ne doit pas entrer dans les moyennes de Recall@k, faute de sources pertinentes attendues.

Le contenu est incorporé à ce notebook afin qu'il s'ouvre directement depuis Colab, sans dépendre d'un chemin de fichier externe.

In [ ]:
import json, re, time, math, sys, platform
from collections import defaultdict
from pathlib import Path
import numpy as np
import pandas as pd

DATA = json.loads(r'''{"documents": [{"id": "IND-01", "titre": "Ligne B — Alarme VB-04", "domaine": "Industrie 4.0", "theme": "maintenance", "texte": "La fiche de maintenance fictive PR-17 concerne une alarme de vibration VB-04 sur le convoyeur de la ligne B. La procédure documentaire demande de consigner l’heure de l’alarme, l’identifiant de la machine et la valeur mesurée. Le responsable habilité vérifie l’historique des alarmes et l’état du dernier entretien. Le document précise que toute intervention physique dépend des consignes de sécurité du site et de l’autorisation du responsable de maintenance. L’assistant documentaire ne doit pas remplacer cette validation.", "reference": "IND-01", "version": "V1 · corpus pédagogique fictif", "type": "Fiche interne fictive"}, {"id": "IND-02", "titre": "Moteur M-12 — Analyse vibratoire", "domaine": "Industrie 4.0", "theme": "capteurs", "texte": "Le rapport fictif VIB-M12 étudie les vibrations du moteur M-12. Les séries temporelles sont collectées toutes les dix minutes et agrégées par heure pour les rapports. Le rapport distingue vibration moyenne, maximum et dispersion. Une augmentation persistante déclenche une inspection documentaire de l’historique, et non un diagnostic automatique de panne. Les mesures doivent être horodatées et associées à la version du capteur.", "reference": "IND-02", "version": "V1 · corpus pédagogique fictif", "type": "Fiche interne fictive"}, {"id": "IND-03", "titre": "Capteur de température T-8 — Métrologie", "domaine": "Industrie 4.0", "theme": "qualité", "texte": "La procédure fictive MET-T8 traite la calibration du capteur de température T-8. Chaque enregistrement de calibration comporte une date, un technicien habilité, une valeur de référence et un écart constaté. Le registre conserve les versions successives au lieu d’écraser l’historique. Pour une mesure actuelle, l’agent logiciel doit interroger la base de mesures autorisée ; un corpus documentaire statique ne suffit pas.", "reference": "IND-03", "version": "V1 · corpus pédagogique fictif", "type": "Fiche interne fictive"}, {"id": "IND-04", "titre": "TRS / OEE — Production", "domaine": "Industrie 4.0", "theme": "performance", "texte": "Le guide fictif PROD-04 définit le taux de rendement synthétique TRS, appelé OEE en anglais, comme le produit disponibilité × performance × qualité. Les trois facteurs sont des proportions comprises entre zéro et un. Pour une disponibilité de 0,90, une performance de 0,95 et une qualité de 0,98, le TRS vaut environ 0,838. Le guide recommande de toujours mentionner la période analysée et la source des données.", "reference": "IND-04", "version": "V1 · corpus pédagogique fictif", "type": "Fiche interne fictive"}, {"id": "IND-05", "titre": "Maintenance prédictive — Guide interne", "domaine": "Industrie 4.0", "theme": "maintenance", "texte": "Le guide fictif PREV-05 décrit une approche de maintenance prédictive utilisant vibration, température et historique des incidents. Le modèle statistique émet des scores de risque, qui ne doivent pas être présentés comme des certitudes. La validation doit vérifier les dérives de capteurs et les changements de conditions de fonctionnement. Les documents décrivent les modèles, tandis que les mesures récentes sont consultées dans un système de supervision.", "reference": "IND-05", "version": "V1 · corpus pédagogique fictif", "type": "Fiche interne fictive"}, {"id": "IND-06", "titre": "Traçabilité des lots — Atelier", "domaine": "Industrie 4.0", "theme": "traçabilité", "texte": "Le document fictif TRACE-06 explique que chaque lot de production possède un identifiant unique. Les évènements de fabrication sont associés au lot, à une machine, à un créneau horaire et à une version de procédure. Lorsqu’une anomalie qualité est signalée, l’équipe cherche les lots concernés et la documentation applicable à la date concernée. Une réponse RAG doit citer la version du document plutôt que mélanger plusieurs révisions.", "reference": "IND-06", "version": "V1 · corpus pédagogique fictif", "type": "Fiche interne fictive"}, {"id": "LOG-01", "titre": "Réception des marchandises — Entrepôt", "domaine": "Logistique et services", "theme": "entrepôt", "texte": "La procédure fictive REC-01 indique que toute réception doit être associée à un bon de livraison et à une commande attendue. En cas d’écart de quantité, l’agent enregistre une anomalie dans le WMS et la transmet au responsable de quai. Le stock n’est mis à jour qu’après la validation prévue par le processus interne. Les documents d’exploitation précisent les rôles ; l’état réel des commandes provient du WMS.", "reference": "LOG-01", "version": "V1 · corpus pédagogique fictif", "type": "Fiche interne fictive"}, {"id": "LOG-02", "titre": "Règle FEFO — Gestion des stocks", "domaine": "Logistique et services", "theme": "stocks", "texte": "Le référentiel fictif STOCK-02 présente la méthode FEFO : First Expired, First Out. Pour les produits soumis à date limite, les unités dont la date d’expiration est la plus proche sont préparées en priorité, sous réserve des contraintes de qualité et de disponibilité. Le texte distingue FEFO et FIFO, qui suit l’ordre d’entrée en stock. L’assistant doit citer la politique de stock effectivement applicable à la catégorie de produits.", "reference": "LOG-02", "version": "V1 · corpus pédagogique fictif", "type": "Fiche interne fictive"}, {"id": "LOG-03", "titre": "Quais de chargement — Ordonnancement", "domaine": "Logistique et services", "theme": "transport", "texte": "Le manuel fictif QUAI-03 décrit l’ordonnancement des quais de chargement. Chaque créneau est lié à un transporteur, à une unité de chargement et à un niveau de priorité. Lorsqu’un camion arrive en retard, la planification est ajustée par un agent habilité dans le TMS. Un RAG documentaire peut expliquer les règles du planning mais ne doit pas inventer l’horaire actuel d’un camion.", "reference": "LOG-03", "version": "V1 · corpus pédagogique fictif", "type": "Fiche interne fictive"}, {"id": "LOG-04", "titre": "Inventaire tournant — WMS", "domaine": "Logistique et services", "theme": "stocks", "texte": "La procédure fictive INV-04 présente l’inventaire tournant : une fraction des emplacements est contrôlée régulièrement sans immobiliser tout l’entrepôt. Les écarts entre stock physique et stock théorique sont enregistrés dans le WMS puis soumis à validation. Les comptages sont rattachés à une date, une zone et un opérateur autorisé. Les réponses fondées sur ce document décrivent la procédure mais pas les quantités réelles.", "reference": "LOG-04", "version": "V1 · corpus pédagogique fictif", "type": "Fiche interne fictive"}, {"id": "LOG-05", "titre": "Livraison — Gestion des incidents", "domaine": "Logistique et services", "theme": "services", "texte": "La politique fictive LIV-05 indique qu’un incident de livraison reçoit un numéro de ticket et une catégorie : retard, colis endommagé ou adresse incorrecte. Le service client consulte le TMS, vérifie la preuve de livraison lorsqu’elle existe et transmet l’incident au responsable concerné. Un modèle de langage peut expliquer les étapes du dossier, mais seul le système opérationnel donne le statut courant.", "reference": "LOG-05", "version": "V1 · corpus pédagogique fictif", "type": "Fiche interne fictive"}, {"id": "LOG-06", "titre": "Retours clients — Logistique inverse", "domaine": "Logistique et services", "theme": "services", "texte": "La procédure fictive RET-06 décrit les retours clients. Le retour est identifié par une autorisation, une référence de commande et un motif. La réception de l’article est distinguée de la décision de remise en stock. Une inspection qualité peut être nécessaire selon l’état du produit. La base documentaire décrit les conditions de traitement ; le statut du retour est maintenu dans le système transactionnel.", "reference": "LOG-06", "version": "V1 · corpus pédagogique fictif", "type": "Fiche interne fictive"}], "questions": [{"question": "Que prévoit le document PR-17 pour l’alarme VB-04 sur le convoyeur ?", "sources_attendues": ["IND-01"]}, {"question": "À quelle fréquence les mesures vibratoires du moteur M-12 sont-elles collectées ?", "sources_attendues": ["IND-02"]}, {"question": "Quels éléments sont consignés lors de la calibration du capteur T-8 ?", "sources_attendues": ["IND-03"]}, {"question": "Comment calcule-t-on le TRS ou OEE ?", "sources_attendues": ["IND-04"]}, {"question": "Quelles sources de données utilise la maintenance prédictive ?", "sources_attendues": ["IND-05"]}, {"question": "Comment la traçabilité des lots prend-elle en compte les versions des procédures ?", "sources_attendues": ["IND-06"]}, {"question": "Que faire lorsqu’une quantité reçue est différente de la commande ?", "sources_attendues": ["LOG-01"]}, {"question": "Quelle différence existe entre FEFO et FIFO ?", "sources_attendues": ["LOG-02"]}, {"question": "Comment est géré un camion en retard sur son créneau de quai ?", "sources_attendues": ["LOG-03"]}, {"question": "En quoi consiste un inventaire tournant ?", "sources_attendues": ["LOG-04"]}, {"question": "Comment catégoriser un incident de livraison dans le processus fictif ?", "sources_attendues": ["LOG-05"]}, {"question": "Quelle différence entre réception d’un retour et remise en stock ?", "sources_attendues": ["LOG-06"]}, {"question": "Comparez la fiche de l’alarme VB-04 et l’analyse vibratoire du moteur M-12 : quelles vérifications documentaires sont prévues ?", "sources_attendues": ["IND-01", "IND-02"]}, {"question": "Comment la règle FEFO se distingue-t-elle des contrôles d’inventaire tournant dans le WMS ?", "sources_attendues": ["LOG-02", "LOG-04"]}]}''')
documents = DATA['documents']
questions = DATA['questions']
by_id = {d['id']: d for d in documents}
questions_sans_preuve = [
    'Combien de palettes sont en stock dans le WMS à 14 h aujourd’hui ?',
    'Quelle est la valeur de vibration relevée en direct sur le moteur M-12 ?'
]
assert len(documents) == 12 and len(questions) == 14
assert all(set(q['sources_attendues']).issubset(by_id) for q in questions)
print(f'{len(documents)} documents, {len(questions)} questions annotées, {len(questions_sans_preuve)} questions hors couverture')
display(pd.DataFrame([{'id': d['id'], 'titre': d['titre'], 'domaine':d['domaine']} for d in documents]))

**Question de méthode.** Une source annotée est un **document** (identifiant IND-01 / LOG-02), tandis que le retriever peut retourner plusieurs fragments du même document. Nous dédupliquons les identifiants avant de calculer les métriques. Cela doit être déclaré explicitement dans le rapport.

## Étape 2 — Définir les métriques de retrieval (10 min)
Pour un ensemble de références pertinentes `G` et une liste classée de documents distincts `R` :

- `Precision@k = |R[:k] ∩ G| / k` lorsque exactement k résultats sont retournés.
- `Recall@k = |R[:k] ∩ G| / |G|` lorsque `|G| > 0`.
- `Hit@k = 1` si une source pertinente apparaît dans les k premiers résultats.
- `RR = 1/rang` du premier document pertinent, ou 0 si aucun n'est trouvé. `MRR` est la moyenne des RR.

Pour les questions sans source pertinente : évaluer **l'abstention et la couverture** séparément.

In [ ]:
def retrieval_metrics(ranked_ids, gold_ids, k=3):
    # ranked_ids doit contenir des IDs de documents distincts.
    assert len(ranked_ids) == len(set(ranked_ids)), 'Dédupliquer les IDs avant les métriques'
    gold = set(gold_ids)
    if not gold:
        raise ValueError('Questions sans source : évaluer séparément l’abstention')
    top = ranked_ids[:k]
    relevant = sum(doc_id in gold for doc_id in top)
    rr = next((1.0 / (i + 1) for i, doc_id in enumerate(ranked_ids) if doc_id in gold), 0.0)
    return {'precision': relevant / len(top) if top else 0.0,
            'recall': relevant / len(gold),
            'hit': float(relevant > 0),
            'rr': rr}

assert retrieval_metrics(['LOG-02','LOG-01','LOG-04','IND-06'], ['LOG-02','LOG-04'],3) == {
    'precision': 2/3, 'recall': 1.0, 'hit': 1.0, 'rr': 1.0}
assert retrieval_metrics(['LOG-01','LOG-02'],['LOG-02'],1)['recall'] == 0
print('Tests des métriques : OK')

## Étape 3 — Variante A : recherche vectorielle de référence (15 min)
Chaque document est utilisé comme un passage unique, comme dans les TP précédents. Le modèle **multilingue** génère un embedding pour chaque document et chaque question. Une normalisation L2 transforme le produit scalaire en similarité cosinus.

La première exécution télécharge le modèle sur Hugging Face (connexion réseau nécessaire).

In [ ]:
import faiss
from sentence_transformers import SentenceTransformer

EMBEDDING_MODEL = 'sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2'
encodeur = SentenceTransformer(EMBEDDING_MODEL)

def encode(texts):
    return np.ascontiguousarray(encodeur.encode(texts, normalize_embeddings=True, show_progress_bar=False), dtype='float32')

def chunks_entiers():
    return [{'chunk_id': d['id']+'-0', 'doc_id':d['id'], 'titre':d['titre'],
             'texte': d['titre']+'. '+d['texte']} for d in documents]

def indexer(chunks):
    vectors = encode([c['texte'] for c in chunks])
    index = faiss.IndexFlatIP(vectors.shape[1])
    index.add(vectors)
    return index

chunks_A = chunks_entiers()
index_A = indexer(chunks_A)
print('Dimensions vecteurs:', index_A.d, '| passages:', index_A.ntotal)

In [ ]:
def dense_ranking(question, chunks, index, n_docs=12):
    """Ordre de documents distincts : score du meilleur chunk, puis déduplication."""
    q = encode([question])
    distances, indices = index.search(q, len(chunks))
    ranked = []
    seen = set()
    for idx in indices[0]:
        d = chunks[int(idx)]['doc_id']
        if d not in seen:
            seen.add(d); ranked.append(d)
        if len(ranked) >= n_docs:
            break
    return ranked

question_test = questions[0]['question']
print('Question:', question_test)
print('Top 5:', dense_ranking(question_test, chunks_A, index_A)[:5])

## Étape 4 — Variante B : effet du chunking (15 min)
Nous divisons chaque fiche en groupes de phrases d'environ 2 phrases. Le même modèle d'embeddings et les mêmes questions sont conservés.

**Attention :** ici, chaque document a 1 à 3 chunks ; l'évaluation reste calculée **par document distinct**. Il s'agit d'une comparaison exploratoire et non d'une preuve que le chunking améliore toujours un système.

In [ ]:
def phrases(texte):
    return [s.strip() for s in re.split(r'(?<=[.!?])\s+', texte.strip()) if s.strip()]

def chunks_par_phrases(group_size=2):
    fragments = []
    for d in documents:
        ps = phrases(d['texte'])
        for i in range(0, len(ps), group_size):
            fragments.append({'chunk_id':f"{d['id']}-{i//group_size}", 'doc_id':d['id'],
                              'titre':d['titre'], 'texte':d['titre']+'. '+' '.join(ps[i:i+group_size])})
    return fragments

chunks_B = chunks_par_phrases(2)
index_B = indexer(chunks_B)
print('Variante A:',len(chunks_A),'chunks ; variante B:',len(chunks_B),'chunks')
print('B top 5:', dense_ranking(question_test, chunks_B, index_B)[:5])

## Étape 5 — Variante C : recherche hybride BM25 + dense (15 min)
Un mot rare comme `VB-04` bénéficie d'une correspondance lexicale exacte. La recherche dense peut retrouver des formulations proches mais sans ce code.

Nous calculons un **BM25 sur les documents entiers**, puis combinons avec le classement dense A au moyen de RRF. Les scores de BM25 et les cosinus ne sont **jamais additionnés directement**.

In [ ]:
from rank_bm25 import BM25Okapi

TOKEN_RE = re.compile(r"[\w]+(?:-[\w]+)*", re.UNICODE)
def tokenize(text):
    return TOKEN_RE.findall(text.casefold())

ordered_ids = [d['id'] for d in documents]
texts = [d['titre']+'. '+d['texte'] for d in documents]
bm25 = BM25Okapi([tokenize(t) for t in texts])

def lexical_ranking(question):
    scores = bm25.get_scores(tokenize(question))
    order = sorted(range(len(documents)), key=lambda i: (-float(scores[i]), i))
    return [ordered_ids[i] for i in order]

def rrf(lists, rank_constant=60):
    scores = defaultdict(float)
    for ranking in lists:
        for i, d in enumerate(ranking, start=1):
            scores[d] += 1/(rank_constant+i)
    return sorted(scores, key=lambda d: (-scores[d], d))

def hybrid_ranking(question):
    return rrf([lexical_ranking(question), dense_ranking(question,chunks_A,index_A)])

print('BM25:',lexical_ranking(question_test)[:5])
print('Dense:',dense_ranking(question_test,chunks_A,index_A)[:5])
print('RRF:',hybrid_ranking(question_test)[:5])
assert sorted(rrf([['a','b'],['b','a']])) == ['a','b']

## Étape 6 — Variante D : réordonnancement interprétable (10 min)
Le **reranking neuronal** le plus connu recalcule souvent la pertinence par un cross-encoder. Pour garantir un atelier exécuté sans GPU ou API, nous utilisons ici un **réordonnancement heuristique**, explicitement distinct d'un cross-encoder.

Il ne voit que les 8 premiers documents de la variante C. Son critère simple privilégie la présence de codes alphanumériques exacts dans la fiche. Une telle règle peut aider ou dégrader le résultat : il faut le mesurer, ne pas le présupposer.

In [ ]:
def technical_codes(text):
    """Exemples : PR-17, VB-04, M-12, T-8, FEFO, FIFO, WMS, TRS, OEE."""
    tokens = tokenize(text)
    return {x for x in tokens if re.search(r'\d',x) or x in {'fefo','fifo','wms','tms','trs','oee'}}

def rerank_heuristic(question, candidates):
    qcodes = technical_codes(question)
    def feature(doc_id):
        d = by_id[doc_id]
        codes = technical_codes(d['titre']+' '+d['texte'])
        return len(qcodes & codes)
    return sorted(candidates, key=lambda doc_id:(-feature(doc_id), candidates.index(doc_id)))

def enhanced_ranking(question, n_candidates=8):
    base = hybrid_ranking(question)
    front = rerank_heuristic(question, base[:n_candidates])
    return front + base[n_candidates:]

print('Hybride:',hybrid_ranking(question_test)[:5])
print('Après règle:',enhanced_ranking(question_test)[:5])

## Étape 7 — Mesurer et comparer les variantes (15 min)
L'évaluation agrège les questions **positives seulement**. Pour obtenir une même base de comparaison, nous demandons 3 documents distincts dans chaque configuration. RR est calculé sur **tout le classement** et non uniquement le top-3. Un temps par requête sera mesuré à titre indicatif : sur un si petit corpus, il est instable et ne reflète pas une charge de production.

In [ ]:
RETRIEVERS = {
 'A — Dense 1 document/chunk': lambda q: dense_ranking(q, chunks_A, index_A),
 'B — Dense chunks courts': lambda q: dense_ranking(q, chunks_B, index_B),
 'C — Hybride BM25 + RRF': hybrid_ranking,
 'D — Hybride + règle métier': enhanced_ranking,
}

def evaluer(retrievers, questions, k=3):
    lignes=[]
    for name, ranker in retrievers.items():
        for ix, item in enumerate(questions):
            start=time.perf_counter()
            ranked=ranker(item['question'])
            ms=(time.perf_counter()-start)*1000
            m=retrieval_metrics(ranked, item['sources_attendues'],k)
            lignes.append({'configuration':name,'q_id':ix+1,'question':item['question'],
                            'attendus':', '.join(item['sources_attendues']),
                            'top_k':', '.join(ranked[:k]),'latence_ms':ms,**m})
    return pd.DataFrame(lignes)

resultats=evaluer(RETRIEVERS, questions, k=3)
resume=(resultats.groupby('configuration',sort=False)
         .agg(precision_at_3=('precision','mean'), recall_at_3=('recall','mean'),
              hit_at_3=('hit','mean'), mrr=('rr','mean'), latence_moy_ms=('latence_ms','mean')))
display(resume.round(3))
assert len(resultats) == 4 * len(questions)
assert resultats[['precision','recall','hit','rr']].to_numpy().min() >= 0
assert resultats[['precision','recall','hit','rr']].to_numpy().max() <= 1
print('Métriques réellement calculées sur les classements : OK')

**Interprétation importante.** Ce tableau est un **résultat à analyser**, pas une compétition à gagner. La variante D peut être plus mauvaise que C ; si c'est le cas, expliquez pourquoi. Le modèle dense et la taille des documents peuvent également limiter les bénéfices du chunking.

In [ ]:
import matplotlib.pyplot as plt
ax=resume[['recall_at_3','mrr']].plot(kind='bar',figsize=(10,4),rot=12)
ax.set_ylim(0,1.02)
ax.set_title('Comparaison expérimentale sur le corpus fictif')
ax.set_ylabel('Score (0 à 1)')
ax.legend(['Recall@3','MRR'])
plt.tight_layout()
plt.show()

## Étape 7 bis — Un jeu plus difficile : questions reformulées (10 min)

Les questions précédentes reprennent souvent les **mots mêmes des fiches** (`PR-17`, `VB-04`, `FEFO`…). Sur un tel jeu, une simple recherche lexicale retrouve déjà presque toutes les sources : les quatre variantes obtiennent des scores voisins et **le banc d'essai ne départage rien**.

Les douze questions ci-dessous visent les mêmes fiches, mais **sans employer leur vocabulaire** : « tapis roulant » pour *convoyeur*, « sonde thermique » pour *capteur de température*, « poids lourd » pour *camion*. C'est la situation réelle d'un utilisateur qui ne connaît pas les termes du document.

On ajoute **BM25 seul** à la comparaison pour isoler ce que la recherche lexicale sait faire sans le modèle dense.

In [ ]:
questions_reformulees = [
    {'question': 'Quelles informations noter quand un tapis roulant signale des secousses anormales ?', 'sources_attendues': ['IND-01']},
    {'question': 'À quel rythme relève-t-on les oscillations d’un entraînement électrique avant de les résumer ?', 'sources_attendues': ['IND-02']},
    {'question': 'Que doit-on garder comme trace lorsqu’on étalonne une sonde thermique ?', 'sources_attendues': ['IND-03']},
    {'question': 'Comment évaluer l’efficacité globale d’un équipement à partir de trois ratios ?', 'sources_attendues': ['IND-04']},
    {'question': 'Peut-on se fier entièrement aux prévisions de défaillance produites par un algorithme ?', 'sources_attendues': ['IND-05']},
    {'question': 'Comment remonter aux fabrications touchées par un défaut signalé ?', 'sources_attendues': ['IND-06']},
    {'question': 'Que se passe-t-il si le nombre de colis déchargés ne correspond pas à ce qui était prévu ?', 'sources_attendues': ['LOG-01']},
    {'question': 'Quels articles périssables faut-il expédier en premier ?', 'sources_attendues': ['LOG-02']},
    {'question': 'Qui modifie l’affectation des portes d’expédition quand un poids lourd n’est pas à l’heure ?', 'sources_attendues': ['LOG-03']},
    {'question': 'Comment compter les articles en rayonnage sans arrêter l’activité du site ?', 'sources_attendues': ['LOG-04']},
    {'question': 'Vers quel outil se tourner quand un client signale un paquet abîmé ?', 'sources_attendues': ['LOG-05']},
    {'question': 'Un produit renvoyé par un acheteur est-il automatiquement revendu ?', 'sources_attendues': ['LOG-06']},
]
assert len(questions_reformulees) == 12 and all(set(q['sources_attendues']) <= set(by_id) for q in questions_reformulees)

RETRIEVERS_PLUS = {'BM25 seul (lexical)': lexical_ranking, **RETRIEVERS}
res_origine = evaluer(RETRIEVERS_PLUS, questions, k=3)
res_reformule = evaluer(RETRIEVERS_PLUS, questions_reformulees, k=3)

def synthese(df):
    return df.groupby('configuration', sort=False).agg(hit_at_3=('hit','mean'), mrr=('rr','mean'))

comparaison = synthese(res_origine).join(synthese(res_reformule), lsuffix=' — questions d’origine', rsuffix=' — reformulées')
display(comparaison.round(3))
ax = comparaison[['hit_at_3 — questions d’origine','hit_at_3 — reformulées']].plot(kind='bar', figsize=(10,4), rot=12)
ax.set_ylim(0,1.02); ax.set_ylabel('Hit@3'); ax.set_title('Même corpus, deux jeux de questions')
ax.legend(['Questions d’origine','Questions reformulées'])
plt.tight_layout(); plt.show()

echecs = res_reformule[(res_reformule.configuration=='BM25 seul (lexical)') & (res_reformule.hit==0)]
print('Questions reformulées manquées par BM25 seul :', len(echecs), '/', len(questions_reformulees))
print('À analyser : lesquelles la recherche dense rattrape-t-elle ? La fusion RRF conserve-t-elle ce gain ?')

## Étape 8 — Diagnostiquer les cas gagnants et perdants (10 min)
Les agrégats masquent les questions problématiques. On inspecte les sources attendues et les classements côte à côte : ce travail est souvent plus instructif que de comparer seulement deux décimales.

In [ ]:
pivot=resultats.pivot(index='q_id',columns='configuration',values='recall')
pivot['question']=[x['question'] for x in questions]
pivot['delta_D_vs_A']=pivot['D — Hybride + règle métier']-pivot['A — Dense 1 document/chunk']
display(pivot[['question','A — Dense 1 document/chunk','C — Hybride BM25 + RRF',
               'D — Hybride + règle métier','delta_D_vs_A']].round(3))

for i in np.argsort(pivot['delta_D_vs_A'].to_numpy())[:3]:
    question=questions[int(i)]['question']
    print('\nQuestion',int(i)+1,question)
    print('Or:',questions[int(i)]['sources_attendues'])
    for name,fn in RETRIEVERS.items():
        print(name,'→',fn(question)[:3])

## Étape 9 — Évaluer séparément une réponse générée (10 min)
Nous n'inventons pas de score de fidélité. Ici, les étudiants annotent **trois affirmations pré-écrites** à partir d'une fiche du corpus. Pour une véritable évaluation automatisée de la génération, reprendre les réponses produites au TP 2 et utiliser un jugement humain, ou un juge LLM calibré et audité (par exemple via Ragas).

**Contexte documentaire :** la fiche PR-17 impose de consigner l'heure, l'identifiant machine et la valeur mesurée ; le responsable habilité consulte l'historique. Aucun seuil de température n'est indiqué.

In [ ]:
affirmations = [
    {'affirmation':'Il faut consigner heure, identifiant et valeur mesurée.', 'etayee':True},
    {'affirmation':'Le responsable habilité vérifie l’historique.', 'etayee':True},
    {'affirmation':'L’arrêt est obligatoire dès 85 °C.', 'etayee':False},
]
# À réaliser avant d'afficher la correction : renseigner sa propre annotation.
annotations_etudiant = [True, True, False]
score_annotation = sum(x == y['etayee'] for x,y in zip(annotations_etudiant,affirmations))/len(affirmations)
fidelite_illustrative=sum(x['etayee'] for x in affirmations)/len(affirmations)
print('Accord des annotations avec le corrigé :',round(score_annotation,3))
print('Part des affirmations étayées dans CET EXEMPLE écrit à la main :',round(fidelite_illustrative,3))
print('ATTENTION : ces chiffres ne sont pas des métriques sur les réponses du LLM du TP 2.')

## Étape 10 — Tester une question sans réponse et produire le compte rendu (5 min)
Un retriever est conçu pour renvoyer des résultats : même lorsque la question n'est pas couverte, il peut renvoyer des passages « les moins mauvais ». La décision d'abstention nécessite des seuils **calibrés**, un contrôle du contexte ou une vérification supplémentaire. Ici, nous inspectons seulement les cas hors couverture et formulons les limites.

In [ ]:
for q in questions_sans_preuve:
    print('\nQUESTION HORS COUVERTURE :',q)
    print('Dense renvoie néanmoins :',dense_ranking(q,chunks_A,index_A)[:3])
    print('Conclusion : les IDs renvoyés ne prouvent pas que le corpus répond à la question.')

resultats.to_csv('tp3_resultats_par_question.csv',index=False,encoding='utf-8-sig')
resume.to_csv('tp3_comparaison_configurations.csv',encoding='utf-8-sig')
print('Exports créés : tp3_resultats_par_question.csv et tp3_comparaison_configurations.csv')

### Travail à rendre — 4 éléments

1. **Tableau** de Precision@3, Recall@3, Hit@3, MRR et temps indicatifs pour les quatre configurations.
2. **Analyse de trois questions** : identifier quel document manque ou quel classement a changé et pourquoi.
3. **Audit de réponse** : distinguer une affirmation étayée d'une affirmation ajoutée sans preuve ; discuter l'abstention.
4. **Proposition technique argumentée** pour un système industriel ou logistique : choisir une configuration en mentionnant le coût, la latence, la validité documentaire et les autorisations.

**Limites à déclarer :** corpus de 12 documents fictifs, 14 questions positives, annotations élaborées pour l'atelier, pas de charge réseau ni de mesure de production. Le contrôle des affirmations pré-écrites ne remplace pas l'évaluation d'une génération LLM réelle.

**Pour aller plus loin (facultatif)** : brancher un cross-encoder réellement multilingue sur le vivier des candidats, le tester sur le même jeu et comparer sa latence ; collecter au moins 50–100 questions réelles anonymisées et annotées pour un protocole plus fiable.

**Complément :** commentez le tableau de l'étape 7 bis. Pourquoi les écarts entre variantes sont-ils plus visibles sur les questions reformulées ? Que conclure sur la manière de construire un jeu de questions d'évaluation ?
